In [ ]:
%%capture
# Install:
# Kaggle environments.
!git clone https://github.com/Kaggle/kaggle-environments.git
!cd kaggle-environments && pip install .

# GFootball environment.
!apt-get update -y
!apt-get install -y libsdl2-gfx-dev libsdl2-ttf-dev

# Make sure that the Branch in git clone and in wget call matches !!
!git clone -b v2.8 https://github.com/google-research/football.git
!mkdir -p football/third_party/gfootball_engine/lib

!wget https://storage.googleapis.com/gfootball/prebuilt_gameplayfootball_v2.8.so -O football/third_party/gfootball_engine/lib/prebuilt_gameplayfootball.so
!cd football && GFOOTBALL_USE_PREBUILT_SO=1 pip3 install .

In [ ]:
# https://github.com/google-research/football/blob/master/gfootball/doc/observation.md
# https://github.com/Kaggle/kaggle-environments/blob/28e75806001c51f1fecbd5d3d8fe7b0aa83f33e5/kaggle_environments/envs/football/helpers.py

In [ ]:
%%writefile submission.py

from kaggle_environments.envs.football.helpers import *
from random import randint
import math

# Function to return closest player and direction
def short_pass():
    pass

# Function to return if player in tight space
def tight_space():
    pass

# Function to return player density
def player_density():
    pass

# Function to calculate distance
def get_distance(pos1,pos2):
    return (((pos1[0]-pos2[0])**2)+((pos1[1]-pos2[1])**2))**0.5

# Function to update snapShot
def set_snapShot(num):
    global snapShot
    snapShot = num
    return

# Function to get snapShot
def get_snapShot():
    global snapShot
    return snapShot

# Movement directions
directions = [
[Action.TopLeft, Action.Top, Action.TopRight],
[Action.Left, Action.Idle, Action.Right],
[Action.BottomLeft, Action.Bottom, Action.BottomRight]]

dirsign = lambda x: 1 if abs(x) < 0.01 else (0 if x < 0 else 2)

# Set game plan parameters
goalRange = 0.75
wingRange = 0.15
snapShot = 0

@human_readable_agent
def agent(obs):

    ################################
    # Functions and logical checks
    ################################

    # Functions

    # Add direction to action
    def sticky_check(action, direction):
        if direction in obs['sticky_actions']:
            return action
        else:
            return direction

    # combo action patterns
    def combo_actions(action1, action2):
        if action1 in obs['sticky_actions']:
            return action2
        else:
            return action1

    # Sticky actions management
    def get_active_sticky_action(obs, exceptions):
#     """ get release action of the first active sticky action, except those in exceptions list """
        release_action = None
        for k in sticky_actions:
            if k not in exceptions and sticky_actions[k] in obs["sticky_actions"]:
                if k == "sprint":
                    release_action = Action.ReleaseSprint
                elif k == "dribble":
                    release_action = Action.ReleaseDribble
                else:
                    release_action = Action.ReleaseDirection
                break
        return release_action

    # Get coordinate distance
    def get_distance_4xy(x1, y1, x2, y2):
#     """ get two-dimensional Euclidean distance, considering y size of the field """
        return math.sqrt((x1 - x2) ** 2 + (y1 * 2.38 - y2 * 2.38) ** 2)

    # Check if player open
    def opp_prox(x, y, front=False):
        xr = x + 0.05
        xl = x - 0.05
        yt = y - 0.05
        yb = y + 0.05
        
        if (x+0.05) > 1:
            xr = 1
        if (x-0.05) < -1:
            xl = -1
        if (y-0.05) < -0.42:
            yt = -0.42
        if (y+0.05) > 0.42:
            yb = 0.42
        
        cnt=0
        for opp in obs['right_team']:
            # If only checking opponents to the front
            if front:
                if opp[0] > x and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                    cnt+=1
                    continue
            # If checking in all directions
            if opp[0] > xl and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                cnt+=1
        
        return cnt
        


    # Logical checks

    controlled_player_pos = obs['left_team'][obs['active']]

    if obs['ball_owned_team'] == 1 or abs(controlled_player_pos[1]) > wingRange:
        set_snapShot(0)

    if get_snapShot() == 1:
        return Action.Shot

    ###############
    # Game modes
    ###############

    # Pass when KickOff or FreeKick or ThrowIn
    if obs['game_mode'] == GameMode.KickOff or obs['game_mode'] == GameMode.ThrowIn:
        return sticky_check(Action.ShortPass, Action.Right)

    # Shoot when freekick in goal range; If on wing then cross; Otherwise just pass
    if obs['game_mode'] == GameMode.FreeKick:
        # Shoot if in range
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingRange:
            ydir = randint(0,2)
            set_snapShot(1)
            return sticky_check(Action.Shot, directions[ydir][2])
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            set_snapShot(1)
            return sticky_check(Action.HighPass, Action.TopRight)

        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            set_snapShot(1)
            return sticky_check(Action.HighPass, Action.BottomRight)

    # Cross in for corner
    if obs['game_mode'] == GameMode.Corner and obs['ball'][1] < 0:
        set_snapShot(1)
        return sticky_check(Action.HighPass, Action.Bottom)
    elif obs['game_mode'] == GameMode.Corner and obs['ball'][1] > 0:
        set_snapShot(1)
        return sticky_check(Action.HighPass, Action.Top)

    # GoalKick
    if obs['game_mode'] == GameMode.GoalKick:
        # Pass to wingbacks if free
        if opp_prox(obs['left_team'][2][0], obs['left_team'][2][1]) == 0:
            return sticky_check(Action.ShortPass, Action.TopRight)
        elif opp_prox(obs['left_team'][3][0], obs['left_team'][3][1]) == 0:
            return sticky_check(Action.ShortPass, Action.BottomRight)
        # Launch downfield
        ydir = randint(0,2)
        return sticky_check(Action.LongPass, directions[ydir][2])

    # Shoot when Penalty
    if obs['game_mode'] == GameMode.Penalty:
        xdir = randint(0,2)
        ydir = randint(0,2)
        set_snapShot(1)
        return sticky_check(Action.Shot, directions[ydir][xdir])

    #####################
    # Sprint or Dribble
    #####################

    # Make sure player is running.
    if Action.Sprint not in obs['sticky_actions']:
        return Action.Sprint
    # We always control left team (observations and actions
    # are mirrored appropriately by the environment).
    controlled_player_pos = obs['left_team'][obs['active']]

    ######################
    # Offensive play
    ######################

    # Check if we are in possession
    if obs['ball_owned_player'] == obs['active'] and obs['ball_owned_team'] == 0:
        
        # Wingbacks to wingers
        if obs['active'] == 2 or obs['active'] == 3:
            return sticky_check(Action.HighPass, Action.Right)

        # Defensive position
        # Clear if we are near our goal
        if controlled_player_pos[0] < -(goalRange):
            return sticky_check(Action.HighPass, Action.Right)


        # Offensive positions
        # Shoot if we are in the final third and not at an acute angle
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingRange:
            ydir = randint(0,2)
            set_snapShot(1)
            
            # Check if teammate in better position
            tmm8s = []
            for i, tmm8 in enumerate(obs['left_team']):
                if i == obs['active']:
                    continue
                if tmm8[0] > (controlled_player_pos[0]-0.02):
                    tmm8s.append(i)
            for tmm8 in tmm8s:
                if opp_prox(obs['left_team'][tmm8][0], obs['left_team'][tmm8][1], front=True) == 0:
                    xdir = dirsign(obs['left_team'][tmm8][0] - controlled_player_pos[0])
                    ydir = dirsign(obs['left_team'][tmm8][1] - controlled_player_pos[1])
                    return sticky_check(Action.ShortPass, directions[ydir][xdir])
                    
            # Take shot otherwise    
            return sticky_check(Action.Shot, directions[ydir][2])
        
        
        # Shoot if goalie out of position
        elif (obs['right_team'][0][0] < 0.83 or abs(obs['right_team'][0][1]) > 0.05) and (controlled_player_pos[0] > 0):
            set_snapShot(1)
            
            # Check if teammate in better position
            tmm8s = []
            for i, tmm8 in enumerate(obs['left_team']):
                if i == obs['active']:
                    continue
                if tmm8[0] > (controlled_player_pos[0]-0.02):
                    tmm8s.append(i)
            for tmm8 in tmm8s:
                if opp_prox(obs['left_team'][tmm8][0], obs['left_team'][tmm8][1], front=True) == 0:
                    xdir = dirsign(obs['left_team'][tmm8][0] - controlled_player_pos[0])
                    ydir = dirsign(obs['left_team'][tmm8][1] - controlled_player_pos[1])
                    return sticky_check(Action.ShortPass, directions[ydir][xdir])
                    
            # Take shot otherwise    
            return Action.Shot
        
        
        # Shoot if attacker very close to goalie
        # Player one on one with goalie
        goalkeeper_x = obs["right_team"][0][0] + obs["right_team_direction"][0][0] * 13
        goalkeeper_y = obs["right_team"][0][1] + obs["right_team_direction"][0][1] * 13
        # player have the ball and located close to the goalkeeper
        if get_distance_4xy(controlled_player_pos[0], controlled_player_pos[1], goalkeeper_x, goalkeeper_y) < 0.25:
            set_snapShot(1)
            
            # Check if teammate in better position
            tmm8s = []
            for i, tmm8 in enumerate(obs['left_team']):
                if i == obs['active']:
                    continue
                if tmm8[0] > (controlled_player_pos[0]-0.02):
                    tmm8s.append(i)
            for tmm8 in tmm8s:
                if opp_prox(obs['left_team'][tmm8][0], obs['left_team'][tmm8][1], front=True) == 0:
                    xdir = dirsign(obs['left_team'][tmm8][0] - controlled_player_pos[0])
                    ydir = dirsign(obs['left_team'][tmm8][1] - controlled_player_pos[1])
                    return sticky_check(Action.ShortPass, directions[ydir][xdir])
                    
            # Take shot otherwise    
            return Action.Shot


        # Crossing positions
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            set_snapShot(1)
            return sticky_check(Action.HighPass, Action.TopRight)

        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            set_snapShot(1)
            return sticky_check(Action.HighPass, Action.BottomRight)

        # Stay within play area
        if controlled_player_pos[0] > 0.95:
            return Action.Left
        elif controlled_player_pos[1] > 0.4:
            return Action.Bottom
        elif controlled_player_pos[1] < -0.4:
            return Action.Top
        # Run towards the goal otherwise.
        return Action.Right


    ###################
    # Defensive play
    ###################

    elif get_snapShot() != 1:
        #where ball is going we add the direction xy to ball current location
        ball_targetx=obs['ball'][0]+(10 * obs['ball_direction'][0])
        ball_targety=obs['ball'][1]+(10 * obs['ball_direction'][1])

        # Euclidian distance to ball
        e_dist=get_distance(obs['left_team'][obs['active']],obs['ball'])

        if e_dist >.005:
            # Run where ball will be
            xdir = dirsign(ball_targetx - controlled_player_pos[0])
            ydir = dirsign(ball_targety - controlled_player_pos[1])
            return directions[ydir][xdir]
        else:
            prob = randint(0,100)
            if prob > 50 and controlled_player_pos[0] < obs['right_team'][obs['ball_owned_player']][0]:
                return Action.Slide
            # Run towards the ball.
            xdir = dirsign(obs['ball'][0] - controlled_player_pos[0])
            ydir = dirsign(obs['ball'][1] - controlled_player_pos[1])
            return directions[ydir][xdir]


In [ ]:
# use dirsign to shoot to the wrong side of the goalie
# If in final third / about to run into player Release Sprint, Dribble
# If lost possession ReleaseDribble, Sprint

In [ ]:
# Set up the Environment.
from kaggle_environments import make
env = make("football", debug=True, configuration={"save_video": True, "scenario_name": "11_vs_11_kaggle", "running_in_notebook": True})
output = env.run(["/kaggle/working/submission.py", "do_nothing"])[-1]
print('Left player: reward = %s, status = %s, info = %s' % (output[0]['reward'], output[0]['status'], output[0]['info']))
print('Right player: reward = %s, status = %s, info = %s' % (output[1]['reward'], output[1]['status'], output[1]['info']))
env.render(mode="human", width=800, height=600)

In [ ]:
# Find player density on pitch to pass to open space / run into open space
# Dribbling
# Shoot if goalie off the line